# v34 CPU all-route audit and rule sweep

This notebook intentionally tries many **CPU-only** routes over the current v32.2 baseline. It is for broad exploration across accounts, but it does **not** select a new best unless a candidate passes strict verification.

Routes included:
- No→Yes textual/Horn-like rule variants with different thresholds.
- Conservative special-case patterns: direct premise support, explicit statement true, “from premise … therefore …”.
- Optional pool-based ablations if `v23_val_candidates.json` is present.

Safety:
- Baseline = v32.2 if present.
- Does not touch MC.
- Does not touch banked indices `{14,71,109,125}`.
- Saves all candidates and a leaderboard.
- The best candidate is only exported as `v34_cpu_best_*` if it beats v32.2 and passes gates.

In [ ]:
import json, re, math, os, sys, random, itertools, statistics
from pathlib import Path
from collections import Counter, defaultdict

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
]

def find_file(names, required=True):
    if isinstance(names, str): names = [names]
    for d in CANDIDATE_DIRS:
        for name in names:
            p = d / name
            if p.exists(): return p
    for d in CANDIDATE_DIRS:
        if d.exists():
            for root, dirs, files in os.walk(d):
                for name in names:
                    if name in files:
                        return Path(root)/name
    if required: raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None: return None, None
    with open(p, 'r', encoding='utf-8') as f: return json.load(f), p

def save_json(obj, name):
    p = Path('/kaggle/working')/name
    try: p.parent.mkdir(parents=True, exist_ok=True)
    except Exception: pass
    with open(p, 'w', encoding='utf-8') as f: json.dump(obj, f, ensure_ascii=False, indent=2)
    print('saved', p)
    return p

LABELS = ['A','B','C','D','Yes','No','Unknown']

def safe_pred(x):
    return x if x in LABELS else 'UNPARSEABLE'

def prf_for_label(rows, lab):
    tp=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')==lab)
    fp=sum(1 for r in rows if r.get('gold')!=lab and r.get('pred')==lab)
    fn=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')!=lab)
    prec=tp/(tp+fp) if tp+fp else 0.0
    rec=tp/(tp+fn) if tp+fn else 0.0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
    return {'precision':prec,'recall':rec,'f1':f1,'support':tp+fn,'pred_count':tp+fp}

def metrics(rows):
    n=len(rows); acc=sum(1 for r in rows if r.get('gold')==r.get('pred'))/n
    per={lab: prf_for_label(rows, lab) for lab in LABELS}
    macro=sum(per[lab]['f1'] for lab in LABELS)/len(LABELS)
    weighted=sum(per[lab]['f1']*per[lab]['support'] for lab in LABELS)/sum(per[lab]['support'] for lab in LABELS)
    mc=[r for r in rows if r.get('is_mc')]
    ynu=[r for r in rows if not r.get('is_mc')]
    mc_labs=['A','B','C','D']; ynu_labs=['Yes','No','Unknown']
    mc_macro=sum(prf_for_label(mc, lab)['f1'] for lab in mc_labs)/4 if mc else 0
    ynu_macro=sum(prf_for_label(ynu, lab)['f1'] for lab in ynu_labs)/3 if ynu else 0
    return {'n':n,'acc':acc,'macro_f1':macro,'weighted_f1':weighted,'mc_macro':mc_macro,'ynu_macro':ynu_macro,'per_label':per,
            'gold_count':dict(Counter(r.get('gold') for r in rows)), 'pred_count':dict(Counter(r.get('pred') for r in rows))}

def diffs(a,b):
    return [r1['idx'] for r1,r2 in zip(a,b) if r1.get('pred')!=r2.get('pred')]

def clone_rows(rows): return [dict(r) for r in rows]

def load_baselines():
    v27, p27 = load_json('v27_standard_preds.json', required=False)
    base, pb = load_json(['v32_2_full_preds.json','v32_2_independent_full_preds.json','v32_2_standard_preds.json','v30_1_full_preds.json'], required=True)
    print('baseline path:', pb)
    base_m = metrics(base)
    print('baseline macro:', base_m['macro_f1'])
    return v27, p27, base, pb, base_m

BANKED = {14,71,109,125}
BASELINE_MACRO = 0.596858548901435

In [ ]:
v27, p27, base, pb, base_m = load_baselines()
assert abs(base_m['macro_f1'] - BASELINE_MACRO) < 1e-9, f"Expected v32.2 baseline; got {base_m['macro_f1']}"
print('READY: v32.2 baseline verified')

In [ ]:
# Feature extraction over explanation text. This is intentionally broad for sweep/audit, not final proof.
def extract_features(r):
    q = str(r.get('question',''))
    ex = str(r.get('explanation',''))
    txt = (q + '\n' + ex).lower()
    body = re.sub(r'final answer\s*[:\-]?\s*(yes|no|unknown|[abcd])\b.*', '', ex, flags=re.I|re.S).lower()
    support = re.findall(r'(?:premise|premises|supporting premises)\s*[:\[]?\s*([0-9,\s]+)', ex, flags=re.I)
    support_nums = set()
    for s in support:
        for n in re.findall(r'\d+', s): support_nums.add(int(n))
    final_m = re.findall(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', ex, flags=re.I)
    old_final = final_m[-1].title() if final_m else None
    if old_final in ['A','B','C','D']: old_final = old_final.upper()
    horn_cues = sum(txt.count(w) for w in ['if ', 'then', 'implies', 'imply', 'leads to', 'from premise', 'therefore', 'thus', 'hence'])
    affirm_cues = sum(txt.count(w) for w in ['therefore', 'thus', 'qualifies', 'can ', 'is true', 'the statement is true', 'does follow', 'complete pathway', 'all ', 'every ', 'exists'])
    neg_window = body[-450:]
    negative_near = bool(re.search(r'false|not true|cannot|does not follow|insufficient|unknown|uncertain|not supported|fails to|denying|disqualifying', neg_window))
    body_true = bool(re.search(r'(statement is true|therefore[^.]{0,120}(true|qualif|can|follow|validated|optimized|complete|meets)|thus[^.]{0,120}(true|qualif|can|follow|validated|optimized|complete|meets))', body))
    direct_positive = bool(re.search(r'(premise \d+ (states|says)[^.]{0,180}(qualif|can|is true|allows|completed|every|all|exists|therefore))', body))
    return dict(
        support_count=len(support_nums), old_inner_final=old_final, horn_cues=horn_cues,
        affirm_cues=affirm_cues, negative_near=negative_near, body_true=body_true,
        direct_positive=direct_positive, text_len=len(txt)
    )

def candidate_indices(rows, params):
    out=[]
    for r in rows:
        if r.get('idx') in BANKED: continue
        if r.get('is_mc'): continue
        if r.get('pred') != 'No': continue
        f=extract_features(r)
        ok=True
        if f['support_count'] < params.get('min_support',0): ok=False
        if f['horn_cues'] < params.get('min_horn',0): ok=False
        if f['affirm_cues'] < params.get('min_affirm',0): ok=False
        if params.get('require_body_true') and not f['body_true']: ok=False
        if params.get('require_direct_positive') and not f['direct_positive']: ok=False
        if params.get('block_negative_near') and f['negative_near']: ok=False
        if params.get('require_old_no') and f['old_inner_final'] != 'No': ok=False
        if ok: out.append(r['idx'])
    return out

def apply_flips(rows, idxs, name):
    new=clone_rows(rows)
    idxset=set(idxs)
    for r in new:
        if r['idx'] in idxset:
            r['pred']='Yes'
            r['repair']=name
    return new

param_grid=[]
for min_support in [1,2,3,4,5]:
  for min_horn in [1,2,3,4,5,6]:
    for min_affirm in [1,2,3,4]:
      for require_body_true in [False, True]:
        for require_direct_positive in [False, True]:
          for block_negative_near in [True]:
            param_grid.append(dict(min_support=min_support,min_horn=min_horn,min_affirm=min_affirm,
                                   require_body_true=require_body_true, require_direct_positive=require_direct_positive,
                                   block_negative_near=block_negative_near, require_old_no=True))
print('grid size', len(param_grid))

In [ ]:
leader=[]
for k,params in enumerate(param_grid):
    idxs=candidate_indices(base, params)
    if not idxs: continue
    cand=apply_flips(base, idxs, f'v34_cpu_sweep_{k}')
    m=metrics(cand)
    per=m['per_label']
    leader.append({
        'variant':f'cpu_sweep_{k}', 'params':params, 'n_flips':len(idxs), 'flips':idxs,
        'macro_f1':m['macro_f1'], 'acc':m['acc'], 'ynu_macro':m['ynu_macro'], 'mc_macro':m['mc_macro'],
        'yes_f1':per['Yes']['f1'], 'no_f1':per['No']['f1'], 'unknown_f1':per['Unknown']['f1'],
        'yes_pred_count':per['Yes']['pred_count'], 'no_pred_count':per['No']['pred_count'],
        'delta_macro':m['macro_f1']-base_m['macro_f1'],
        'posthoc_gold':dict(Counter(r['gold'] for r in base if r['idx'] in set(idxs)))
    })
leader=sorted(leader, key=lambda x:x['macro_f1'], reverse=True)
print('variants tried:', len(leader))
print('top 15')
for row in leader[:15]:
    print(row['variant'], 'macro', row['macro_f1'], 'delta', row['delta_macro'], 'n', row['n_flips'], 'gold', row['posthoc_gold'], 'params', row['params'])
save_json({'version':'v34_cpu_rule_sweep_leaderboard','baseline_macro':base_m['macro_f1'],'n_variants':len(leader),'leaderboard':leader}, 'v34_cpu_rule_sweep_leaderboard.json')

In [ ]:
# Strict selector for CPU sweep. Requires a real gain and no collapse.
selected=None
if leader:
    top=leader[0]
    gain=top['delta_macro']
    collapse = top['yes_pred_count'] > 45 or top['no_pred_count'] < 30 or top['no_f1'] < base_m['per_label']['No']['f1'] - 0.03
    if gain > 0.01 and not collapse and top['mc_macro'] == base_m['mc_macro'] and top['unknown_f1'] >= base_m['per_label']['Unknown']['f1']:
        selected=top

if selected:
    best=apply_flips(base, selected['flips'], 'v34_cpu_best_selected')
    best_m=metrics(best)
    save_json(best, 'v34_cpu_best_preds.json')
    reloaded,_=load_json('v34_cpu_best_preds.json')
    assert abs(metrics(reloaded)['macro_f1']-best_m['macro_f1'])<1e-12
    summary={'version':'v34_cpu_best','selection_status':'SELECT_V34_CPU','selected':selected,'metrics':best_m,
             'flipped_indices':selected['flips'], 'baseline_v32_2':base_m}
    save_json(summary, 'v34_cpu_best_summary.json')
    print('SELECTED CPU BEST', best_m['macro_f1'])
else:
    save_json(base, 'v34_cpu_best_preds.json')
    summary={'version':'v34_cpu_best','selection_status':'ROLLBACK_V32_2','reason':'no CPU sweep variant passed strict gates','metrics':base_m,'baseline_v32_2':base_m,
             'best_candidate': leader[0] if leader else None}
    save_json(summary, 'v34_cpu_best_summary.json')
    print('ROLLBACK CPU. Best remains v32.2')